# HP Tuning — Round 4 (Epochs) + Final Training
## Continuing from previous account

### What we know so far
```
Round 1 → best rank  : r=32      (val F1=0.8951, +0.0057 vs baseline)
Round 2 → best lr    : lr=5e-4   (val F1=0.8978, +0.0084 vs baseline)
Round 3 → best alpha : alpha=128 (val F1=0.9039, +0.0145 vs baseline)

Known epochs=3 baseline: val F1=0.9039
Round 4 tests: epochs=2 and epochs=5
```

### Plan
1. Run epochs=2 and epochs=5 (epochs=3 already known = 0.9039)
2. Pick best epochs
3. Final training with r=32, lr=5e-4, alpha=128, best_epochs
4. Evaluate on TEST SET (first time)
5. Push to HF Hub

### ⚠️ 2-Step Start
- Run Cell 1 → Kernel → Restart → Run All

In [1]:
!pip install -q -U trl transformers peft accelerate bitsandbytes datasets scikit-learn torchao
import trl, transformers, peft
print(f"TRL          : {trl.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"PEFT         : {peft.__version__}")
print("✅ Ready — now Restart Kernel and Run All")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 93.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 112.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-l

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


TRL          : 1.9.2
Transformers : 5.14.1
PEFT         : 0.20.0
✅ Ready — now Restart Kernel and Run All


In [2]:
import pandas as pd
import numpy as np
import json, os, time, gc, warnings
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")
set_seed(42)

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("✅ HF login OK")
except Exception as e:
    print(f"⚠️  HF login: {e}")

LABEL2ID   = {"positive": 0, "negative": 1, "neutral": 2}
CSV_PATH   = "/kaggle/input/datasets/vijayadi/sentiment-stocks/SEntFiN-v1.1.csv"
SAVE_DIR   = "/kaggle/working"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

# ── Best HPs from Rounds 1-3 (hardcoded from previous account) ────────
BEST_RANK  = 32
BEST_LR    = 5e-4
BEST_ALPHA = 128

# Known results from previous account
BASELINE_VAL_F1  = 0.8894   # original model
BASELINE_TEST_F1 = 0.9006   # original model
EP3_VAL_F1       = 0.9039   # r=32, lr=5e-4, alpha=128, epochs=3

HP_RESULTS = [
    # Seed with known ep=3 result so leaderboard is complete
    {"split": "r32_lr5e4_a128_ep3", "rank": 32, "lr": 5e-4, "alpha": 128,
     "epochs": 3, "accuracy": 0.9028, "macro_f1": EP3_VAL_F1,
     "neg_f1": 0.9089, "vs_baseline": EP3_VAL_F1 - BASELINE_VAL_F1}
]

print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"\nBest HPs from Rounds 1-3:")
print(f"  rank={BEST_RANK} | lr={BEST_LR} | alpha={BEST_ALPHA}")
print(f"  epochs=3 (known) → val F1={EP3_VAL_F1}")
print(f"  Now testing: epochs=2 and epochs=5")

⚠️  HF login: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 129578414 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.
GPU : Tesla T4
VRAM: 15.6 GB

Best HPs from Rounds 1-3:
  rank=32 | lr=0.0005 | alpha=128
  epochs=3 (known) → val F1=0.9039
  Now testing: epochs=2 and epochs=5


In [3]:
def preprocess_data(csv_path=CSV_PATH, seed=42):
    df = pd.read_csv(csv_path)
    rows = []
    for _, row in df.iterrows():
        try:
            for entity, label in json.loads(row["Decisions"]).items():
                if label in LABEL2ID:
                    rows.append({"headline": str(row["Title"]).strip(),
                                 "entity":   str(entity).strip(),
                                 "label":    label})
        except Exception:
            continue
    data = pd.DataFrame(rows)
    print(f"Exploded: {len(data)} | {data['label'].value_counts().to_dict()}")
    train, temp = train_test_split(data, test_size=0.2, random_state=seed, stratify=data["label"])
    val, test   = train_test_split(temp, test_size=0.5, random_state=seed, stratify=temp["label"])
    print(f"Train {len(train)} | Val {len(val)} | Test {len(test)}")
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

train_df, val_df, test_df = preprocess_data()

Exploded: 14409 | {'neutral': 5515, 'positive': 5075, 'negative': 3819}
Train 11527 | Val 1441 | Test 1441


In [4]:
SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def make_training_text(headline, entity, label, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Headline: {headline}\nCompany: {entity}\nSentiment:"},
        {"role": "assistant", "content": label}
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        msgs = [
            {"role": "user",      "content": f"{SYSTEM_PROMPT}\n\nHeadline: {headline}\nCompany: {entity}\nSentiment:"},
            {"role": "assistant", "content": label}
        ]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def make_inference_prompt(headline, entity, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}\nCompany: {entity}\nSentiment:"}
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        msgs = [{"role": "user",
                 "content": f"{SYSTEM_PROMPT}\n\nHeadline: {headline}\nCompany: {entity}\nSentiment:"}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def extract_label(text):
    t = text.lower().strip()
    for lbl in ["positive", "negative", "neutral"]:
        if t.startswith(lbl) or lbl in t:
            return lbl
    return "neutral"

def evaluate_model(model, tokenizer, df, split_label, batch_size=8):
    model.eval()
    tokenizer.padding_side = "left"
    all_preds, all_labels = [], df["label"].tolist()
    t0 = time.time()

    for i in range(0, len(df), batch_size):
        b       = df.iloc[i:i+batch_size]
        prompts = [make_inference_prompt(r["headline"], r["entity"], tokenizer)
                   for _, r in b.iterrows()]
        inputs  = tokenizer(prompts, return_tensors="pt", padding=True,
                            truncation=True, max_length=120).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        il = inputs["input_ids"].shape[1]
        for seq in out:
            all_preds.append(extract_label(tokenizer.decode(seq[il:], skip_special_tokens=True)))
        if i % (batch_size * 20) == 0:
            print(f"  Eval: {min(i+batch_size, len(df))}/{len(df)}")

    ms     = (time.time() - t0) * 1000 / len(df)
    pred_i = [LABEL2ID.get(p, 2) for p in all_preds]
    lab_i  = [LABEL2ID.get(l, 2) for l in all_labels]
    acc    = accuracy_score(lab_i, pred_i)
    mf1    = f1_score(lab_i, pred_i, average="macro",      zero_division=0)
    per_f1 = f1_score(lab_i, pred_i, average=None, labels=[0,1,2], zero_division=0)

    print(f"\n[{split_label}] Acc={acc:.4f} | MacroF1={mf1:.4f} | "
          f"Pos={per_f1[0]:.4f} Neg={per_f1[1]:.4f} Neu={per_f1[2]:.4f} | {ms:.1f}ms/sample")
    print(classification_report(lab_i, pred_i,
          target_names=["positive","negative","neutral"], zero_division=0))
    return {
        "split": split_label, "accuracy": round(acc,4), "macro_f1": round(mf1,4),
        "pos_f1": round(per_f1[0],4), "neg_f1": round(per_f1[1],4),
        "neu_f1": round(per_f1[2],4), "inference_ms": round(ms,2)
    }

def run_epoch_experiment(epochs, tag):
    print(f"\n{'='*65}")
    print(f"EXPERIMENT: {tag}")
    print(f"  rank={BEST_RANK} | lr={BEST_LR} | alpha={BEST_ALPHA} | epochs={epochs}")
    print(f"{'='*65}")
    t_start = time.time()

    tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token; tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )
    lora_cfg = LoraConfig(
        r=BEST_RANK, lora_alpha=BEST_ALPHA,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    texts = [{"text": make_training_text(r["headline"], r["entity"], r["label"], tok)}
             for _, r in train_df.iterrows()]

    SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=f"{SAVE_DIR}/{tag}",
            dataset_text_field="text", max_length=128,
            packing=False, loss_type="nll",
            num_train_epochs=epochs,
            per_device_train_batch_size=4, gradient_accumulation_steps=8,
            learning_rate=BEST_LR, lr_scheduler_type="cosine", warmup_steps=50,
            fp16=True, bf16=False, logging_steps=50, save_strategy="no",
            optim="adamw_torch", report_to="none",
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        ),
        train_dataset=Dataset.from_list(texts),
        processing_class=tok,
    ).train()

    tok.padding_side = "left"
    result = evaluate_model(model, tok, val_df, split_label=tag)
    result.update({
        "rank": BEST_RANK, "lr": BEST_LR, "alpha": BEST_ALPHA, "epochs": epochs,
        "train_time_min": round((time.time()-t_start)/60, 1),
        "vs_baseline": round(result["macro_f1"] - BASELINE_VAL_F1, 4)
    })
    print(f"  vs baseline: {'+' if result['vs_baseline']>=0 else ''}{result['vs_baseline']:.4f} F1")
    print(f"  vs ep3:      {'+' if result['macro_f1']-EP3_VAL_F1>=0 else ''}{result['macro_f1']-EP3_VAL_F1:.4f} F1")

    del model, tok; gc.collect(); torch.cuda.empty_cache()
    print(f"  GPU cleared | Time: {result['train_time_min']} min")
    return result

## Round 4 — Vary Epochs
**Expected: ~2 hrs** | epochs=3 already known (val F1=0.9039)

In [ ]:
# ══════════════════════════════════════════════════════════════
# ROUND 4 — Vary EPOCHS  (r=32, lr=5e-4, alpha=128)
# epochs=3 already known: val F1=0.9039
# Testing epochs=2 (less training) and epochs=5 (more training)
# Expected: ~2 hrs total
# ══════════════════════════════════════════════════════════════
print("ROUND 4: Varying epochs")
print(f"  Known: epochs=3 → val F1={EP3_VAL_F1}")
print(f"  Testing: epochs=2 (underfitting check) and epochs=5 (overfitting check)")
print()

HP_RESULTS.append(run_epoch_experiment(epochs=2, tag="r32_lr5e4_a128_ep2"))
HP_RESULTS.append(run_epoch_experiment(epochs=5, tag="r32_lr5e4_a128_ep5"))

# ── Round 4 leaderboard ────────────────────────────────────────
df_r4 = pd.DataFrame(HP_RESULTS).sort_values("macro_f1", ascending=False)
print("\n" + "="*70)
print("  ROUND 4 — Epoch Sensitivity (complete)")
print("="*70)
print(df_r4[["split","epochs","accuracy","macro_f1","neg_f1","vs_baseline"]].to_string(index=False))

best_row    = df_r4.iloc[0]
best_epochs = int(best_row["epochs"])
best_val_f1 = float(best_row["macro_f1"])

print(f"\n🏆 Best epochs: {best_epochs} | val F1={best_val_f1:.4f}")
print(f"   Improvement over baseline: +{best_val_f1 - BASELINE_VAL_F1:.4f}")
print(f"\n→ Final training: r={BEST_RANK} | lr={BEST_LR} | alpha={BEST_ALPHA} | epochs={best_epochs}")

## Final Training + Test Set Evaluation
**TEST SET touched only once here**

In [5]:
best_epochs = 3

In [7]:
# ══════════════════════════════════════════════════════════════
# FINAL TRAINING — All best HPs
# TEST SET touched for the first and only time here
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("FINAL TRAINING — Best HPs from all 4 rounds")
print("="*65)
print(f"  rank={BEST_RANK} | lr={BEST_LR} | alpha={BEST_ALPHA} | epochs={best_epochs}")

print(f"  TEST SET will be evaluated once after this training")
print("="*65)

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token; tok.pad_token_id = tok.eos_token_id
tok.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True
)

lora_cfg = LoraConfig(
    r=BEST_RANK, lora_alpha=BEST_ALPHA,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

texts = [{"text": make_training_text(r["headline"], r["entity"], r["label"], tok)}
         for _, r in train_df.iterrows()]

SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=f"{SAVE_DIR}/Qwen_HP_final",
        dataset_text_field="text", max_length=128,
        packing=False, loss_type="nll",
        num_train_epochs=best_epochs,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,
        learning_rate=BEST_LR, lr_scheduler_type="cosine", warmup_steps=50,
        fp16=True, bf16=False, logging_steps=50, save_strategy="no",
        optim="adamw_torch", report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    ),
    train_dataset=Dataset.from_list(texts),
    processing_class=tok,
).train()

model.save_pretrained(f"{SAVE_DIR}/Qwen_HP_final_adapter")
tok.save_pretrained(f"{SAVE_DIR}/Qwen_HP_final_adapter")
print(f"Adapter saved → {SAVE_DIR}/Qwen_HP_final_adapter")

# ── Evaluate on val AND test ───────────────────────────────────
tok.padding_side = "left"
final_val  = evaluate_model(model, tok, val_df,  "Qwen_HP_final_val")
final_test = evaluate_model(model, tok, test_df, "Qwen_HP_final_test")

# ── Final summary ──────────────────────────────────────────────
print("\n" + "="*65)
print("COMPLETE HP TUNING SUMMARY")
print("="*65)
summary = pd.DataFrame([
    {"model": "Original Qwen2.5-3B",
     "HPs": "r=16,lr=2e-4,alpha=32,ep=3",
     "val_f1": BASELINE_VAL_F1, "test_f1": BASELINE_TEST_F1},
    {"model": "HP-Tuned Qwen2.5-3B",
     "HPs": f"r={BEST_RANK},lr={BEST_LR},alpha={BEST_ALPHA},ep={best_epochs}",
     "val_f1": final_val["macro_f1"], "test_f1": final_test["macro_f1"]},
])
summary["test_gain"] = (summary["test_f1"] - BASELINE_TEST_F1).apply(
    lambda x: f"+{x:.4f}" if x >= 0 else f"{x:.4f}"
)
print(summary.to_string(index=False))

print(f"\n  Final Test Accuracy : {final_test['accuracy']:.4f}")
print(f"  Final Test Macro F1 : {final_test['macro_f1']:.4f}")
print(f"  Final Test Neg F1   : {final_test['neg_f1']:.4f}")
print(f"  Final Test Pos F1   : {final_test['pos_f1']:.4f}")
print(f"  Final Test Neu F1   : {final_test['neu_f1']:.4f}")


FINAL TRAINING — Best HPs from all 4 rounds
  rank=32 | lr=0.0005 | alpha=128 | epochs=3
  TEST SET will be evaluated once after this training


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Truncating train dataset:   0%|          | 0/11527 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/11527 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
50,1.095413
100,0.795150
150,0.894304
200,0.955787
250,0.835251
300,0.620534
350,0.597232
400,0.537186
450,0.509279
500,0.495522


Adapter saved → /kaggle/working/Qwen_HP_final_adapter
  Eval: 8/1441
  Eval: 168/1441
  Eval: 328/1441
  Eval: 488/1441
  Eval: 648/1441
  Eval: 808/1441
  Eval: 968/1441
  Eval: 1128/1441
  Eval: 1288/1441
  Eval: 1441/1441

[Qwen_HP_final_val] Acc=0.9022 | MacroF1=0.9025 | Pos=0.9214 Neg=0.8992 Neu=0.8869 | 59.1ms/sample
              precision    recall  f1-score   support

    positive       0.93      0.91      0.92       508
    negative       0.91      0.89      0.90       382
     neutral       0.87      0.90      0.89       551

    accuracy                           0.90      1441
   macro avg       0.90      0.90      0.90      1441
weighted avg       0.90      0.90      0.90      1441

  Eval: 8/1441
  Eval: 168/1441
  Eval: 328/1441
  Eval: 488/1441
  Eval: 648/1441
  Eval: 808/1441
  Eval: 968/1441
  Eval: 1128/1441
  Eval: 1288/1441
  Eval: 1441/1441

[Qwen_HP_final_test] Acc=0.9042 | MacroF1=0.9047 | Pos=0.9166 Neg=0.9043 Neu=0.8932 | 59.0ms/sample
              precisio

In [9]:
# ── Save all results CSV ──────────────────────────────────────
all_results = pd.DataFrame(HP_RESULTS)
all_results.to_csv(f"{SAVE_DIR}/hp_tuning_all_results.csv", index=False)

final_df = pd.DataFrame([final_val, final_test])
final_df.to_csv(f"{SAVE_DIR}/final_results_hp_tuned.csv", index=False)

print(f"✅ HP results → {SAVE_DIR}/hp_tuning_all_results.csv")
print(f"✅ Final results → {SAVE_DIR}/final_results_hp_tuned.csv")

# ── Push to HuggingFace Hub ────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO_NAME = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"
print(f"\nPushing to {REPO_NAME} ...")
model.push_to_hub(REPO_NAME, private=True)
tok.push_to_hub(REPO_NAME, private=True)
print(f"✅ Saved → https://huggingface.co/{REPO_NAME}")

del model, tok
gc.collect(); torch.cuda.empty_cache()
print("\n✅ All done! HP tuning complete.")
print(f"   Best model: r={BEST_RANK}, lr={BEST_LR}, alpha={BEST_ALPHA}, epochs={best_epochs}")

✅ HP results → /kaggle/working/hp_tuning_all_results.csv
✅ Final results → /kaggle/working/final_results_hp_tuned.csv

Pushing to vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Saved → https://huggingface.co/vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned

✅ All done! HP tuning complete.
   Best model: r=32, lr=0.0005, alpha=128, epochs=3


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load from HF Hub
REPO     = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"
BASE     = "Qwen/Qwen2.5-3B-Instruct"

tok  = AutoTokenizer.from_pretrained(REPO)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, REPO)
model.eval()
print("✅ Model loaded")

SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def predict_sentiment(headline, company):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}\nCompany: {company}\nSentiment:"}
    ]
    prompt  = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs  = tok(prompt, return_tensors="pt", truncation=True, max_length=120).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(new_tokens, skip_special_tokens=True).lower().strip()

    for lbl in ["positive", "negative", "neutral"]:
        if raw.startswith(lbl) or lbl in raw:
            return lbl
    return "neutral"

# ── Test it ───────────────────────────────────────────────────
test_cases = [
    ("TCS opens new AI lab in Bengaluru with Nvidia",    "TCS"),
    ("Infosys cuts revenue forecast amid slowdown",      "Infosys"),
    ("Reliance posts record quarterly profit",            "Reliance"),
    ("Wipro bags 300 million dollar cloud deal",         "Wipro"),
    ("Stock falls after weak quarterly earnings",        "Stock"),
]

print(f"\n{'Headline':<55} {'Company':<12} {'Sentiment'}")
print("-"*80)
for headline, company in test_cases:
    pred = predict_sentiment(headline, company)
    print(f"{headline:<55} {company:<12} {pred}")

/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# ── Login (needed since model is private) ─────────────────────────────
from huggingface_hub import login
login()   # will prompt for your HF token, or pass token="hf_xxx" directly

# ── Load from HF Hub ───────────────────────────────────────────────────
REPO     = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"
BASE     = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(REPO)

print("Loading base model (~6GB, first time only)...")
base = AutoModelForCausalLM.from_pretrained(
    BASE,
    dtype=torch.float16,
    device_map="mps"     # Apple Silicon — change to "cpu" if issues
)

print("Loading adapter...")
model = PeftModel.from_pretrained(base, REPO)
model.eval()
print("✅ Model ready\n")

# ── Predict function ───────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def predict(headline, company):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}\nCompany: {company}\nSentiment:"}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=120).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id
        )
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).lower().strip()
    for lbl in ["positive", "negative", "neutral"]:
        if raw.startswith(lbl) or lbl in raw:
            return lbl
    return "neutral"

# ── Tests ──────────────────────────────────────────────────────────────
test_cases = [
    ("TCS opens new AI lab in Bengaluru with Nvidia",         "TCS",      "positive/neutral"),
    ("Infosys expands to 3 new cities across Southeast Asia", "Infosys",  "positive/neutral"),
    ("Wipro bags 300 million dollar cloud deal",              "Wipro",    "positive"),
    ("Reliance posts record quarterly profit",                "Reliance", "positive"),
    ("Infosys cuts revenue forecast amid slowdown",           "Infosys",  "negative"),
    ("Stock falls after weak quarterly earnings",             "Stock",    "negative"),
]

print(f"{'Headline':<55} {'Company':<10} {'Expected':<20} {'Got'}")
print("-"*100)
for headline, company, expected in test_cases:
    pred = predict(headline, company)
    ok   = pred in expected.split("/")
    flag = "✅" if ok else "❌"
    print(f"{flag} {headline:<55} {company:<10} {expected:<20} {pred}")

# ── Interactive ────────────────────────────────────────────────────────
print("\nInteractive mode (Ctrl+C to quit)")
while True:
    try:
        h = input("\nHeadline : ").strip()
        if not h: continue
        c = input("Company  : ").strip() or "Company"
        print(f"→ {predict(h, c).upper()}")
    except KeyboardInterrupt:
        print("\nBye!")
        break

/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...
Loading base model (~6GB, first time only)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# ── Step 1: Login (private model) ─────────────────────────────
from huggingface_hub import login
login()   # paste your HF token when prompted

# ── Step 2: Load (transformers + peft auto classes) ───────────
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch

REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(REPO)

print("Loading model (downloads base ~6GB first time)...")
model = AutoPeftModelForCausalLM.from_pretrained(
    REPO,
    dtype=torch.float16,
    device_map="mps"        # Apple Silicon — change to "cpu" if needed
)
model.eval()
print("✅ Model ready\n")

# ── Step 3: Predict ───────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def predict(headline, company):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}\nCompany: {company}\nSentiment:"}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=120).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id
        )
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).lower().strip()
    for lbl in ["positive", "negative", "neutral"]:
        if raw.startswith(lbl) or lbl in raw:
            return lbl
    return "neutral"

# ── Step 4: Test ──────────────────────────────────────────────
tests = [
    ("TCS opens new AI lab in Bengaluru with Nvidia",         "TCS"),
    ("Infosys cuts revenue forecast amid global slowdown",    "Infosys"),
    ("Reliance posts record quarterly profit",                "Reliance"),
    ("Wipro bags 300 million dollar cloud deal",              "Wipro"),
    ("Stock falls after weak quarterly earnings",             "Stock"),
]

print(f"{'Headline':<55} {'Company':<12} Sentiment")
print("-"*80)
for headline, company in tests:
    print(f"{headline:<55} {company:<12} {predict(headline, company)}")

# ── Step 5: Interactive ───────────────────────────────────────
print("\nType your own (Ctrl+C to quit)")
while True:
    try:
        h = input("\nHeadline : ").strip()
        if not h: continue
        c = input("Company  : ").strip() or "Company"
        print(f"→ {predict(h, c).upper()}")
    except KeyboardInterrupt:
        print("\nBye!")
        break

/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...
Loading model (downloads base ~6GB first time)...


/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:746: UserWarning: Not enough free disk space to download the file. The expected file size is: 2203.27 MB. The target location /Users/vijay/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/blobs only has 437.46 MB free disk space.
  warnings.warn(


In [1]:
# ── Step 1: Login (private model) ─────────────────────────────
from huggingface_hub import login
login()   # paste your HF token when prompted

/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from huggingface_hub import login

login(token="os.environ["HF_TOKEN"]")

In [ ]:
# ── Step 2: Load (transformers + peft auto classes) ───────────
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch

REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(REPO)

print("Loading model (downloads base ~6GB first time)...")
model = AutoPeftModelForCausalLM.from_pretrained(
    REPO,
    dtype=torch.float16,
    device_map="mps"        # Apple Silicon — change to "cpu" if needed
)
model.eval()
print("✅ Model ready\n")

# ── Step 3: Predict ───────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def predict(headline, company):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}\nCompany: {company}\nSentiment:"}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=120).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id
        )
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).lower().strip()
    for lbl in ["positive", "negative", "neutral"]:
        if raw.startswith(lbl) or lbl in raw:
            return lbl
    return "neutral"

# ── Step 4: Test ──────────────────────────────────────────────
tests = [
    ("TCS opens new AI lab in Bengaluru with Nvidia",         "TCS"),
    ("Infosys cuts revenue forecast amid global slowdown",    "Infosys"),
    ("Reliance posts record quarterly profit",                "Reliance"),
    ("Wipro bags 300 million dollar cloud deal",              "Wipro"),
    ("Stock falls after weak quarterly earnings",             "Stock"),
]

print(f"{'Headline':<55} {'Company':<12} Sentiment")
print("-"*80)
for headline, company in tests:
    print(f"{headline:<55} {company:<12} {predict(headline, company)}")

# ── Step 5: Interactive ───────────────────────────────────────
print("\nType your own (Ctrl+C to quit)")
while True:
    try:
        h = input("\nHeadline : ").strip()
        if not h: continue
        c = input("Company  : ").strip() or "Company"
        print(f"→ {predict(h, c).upper()}")
    except KeyboardInterrupt:
        print("\nBye!")
        break

Loading tokenizer...
Loading model (downloads base ~6GB first time)...


/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:746: UserWarning: Not enough free disk space to download the file. The expected file size is: 2203.27 MB. The target location /Users/vijay/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/blobs only has 278.60 MB free disk space.
  warnings.warn(
